<a href="https://colab.research.google.com/github/econ105/AI/blob/main/AI/FINT501/AIagent/yfinance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --quiet langchain-core==0.3.59 pydantic==2.11.9

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 437.7/437.7 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.9/444.9 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 41.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 1.4.0 requires langchain-core<2.0.0,>=1.6.0, but you have langchain-core 0.3.59 which is incompatible.
google-adk 2.7.1 requires pydantic<3,>=2.12, but you have pydantic 2.11.9 which is incompatible.
google-genai 2.12.1 requires pydantic<3.0.0,>=2.12.5, but yo

In [2]:
import os
from typing import Annotated
import pandas as pd
from langchain_core.tools import tool

@tool
def get_stock_performance(
    company_ticker: Annotated[str, "The ticker symbol of the company to retrieve their stock performance data."],
    num_days: Annotated[int, "The number of days of stock data required to respond to the user query."]
) -> str:
    """
    Use this to look-up stock performance data for companies to retrieve a table from a CSV. You may need to convert company names into ticker symbols to call this function, e.g, Apple Inc. -> AAPL, and you may need to convert weeks, months, and years, into days.
    """

    # Load the CSV for the company requested
    file_path = f"data/{company_ticker}.csv"

    if os.path.exists(file_path) is False:
        return f"Sorry, but data for company {company_ticker} is not available. Please try Apple, Amazon, Meta, Microsoft, Netflix, or Tesla."

    stock_df = pd.read_csv(file_path, index_col='Date', parse_dates=True)

    # Ensure the index is in date format
    stock_df.index = stock_df.index.date

    # Maximum num_days supported by the dataset
    max_num_days = (stock_df.index.max() - stock_df.index.min()).days

    if num_days > max_num_days:
        return "Sorry, but this time period exceeds the data available. Please reduce it to continue."

    # Get the most recent date in the DataFrame
    final_date = stock_df.index.max()

    # Filter the DataFrame to get the last num_days of stock data
    filtered_df = stock_df[stock_df.index > (final_date - pd.Timedelta(days=num_days))]

    return f"Successfully executed the stock performance data retrieval tool to retrieve the last *{num_days} days* of data for company **{company_ticker}**:\n\n{filtered_df.to_markdown()}"

In [3]:
# Import necessary library (for example, yfinance)
import yfinance as yf

# Define a function to retrieve stock data
def get_stock_data(company_ticker, num_days):
    stock = yf.Ticker(company_ticker)
    hist = stock.history(period=f"{num_days}d")
    return hist

# Use the function to get data
retrieved_data = get_stock_data("META", 4)
print(retrieved_data)

                                 Open        High         Low       Close  \
Date                                                                        
2026-09-29 00:00:00-04:00  725.000000  740.450012  715.099976  738.789978   
2026-09-30 00:00:00-04:00  730.270020  738.250000  721.299988  725.179993   
2026-10-01 00:00:00-04:00  728.530029  735.880005  721.510010  725.929993   
2026-10-02 00:00:00-04:00  733.250000  741.580017  726.200012  728.080017   

                             Volume  Dividends  Stock Splits  
Date                                                          
2026-09-29 00:00:00-04:00  23591100        0.0           0.0  
2026-09-30 00:00:00-04:00  19293000        0.0           0.0  
2026-10-01 00:00:00-04:00  12408400        0.0           0.0  
2026-10-02 00:00:00-04:00  12944900        0.0           0.0  


Tool 2: Stock performance data
You'll create a tool to load stock performance data from a local CSV file, filter it depending on the user's query, then return the results.

This tool will use stock ticker symbols, which are unique company IDs, to return the relevant information for that company. Here are the companies and symbols that have stock data available in your environment:

Company Name	Ticker Symbol
Apple	AAPL
Microsoft	MSFT
Amazon	AMZN
Meta	META
Netflix	NFLX
Tesla	TSLA